# 03 - Train / Test Split

**Stage:** Load (the *L* in ETL) — load the transformed feature matrix and
split it into reproducible train/test partitions ready for analysis / modelling.

Responsibilities:
- Load the feature matrix from `data/interim/features.parquet` (produced by `02_transform`).
- Split into train/test with a fixed `random_state` for reproducibility.
- **Stratify** on the target label so the beach / not_beach ratio is preserved in both splits.
- Persist splits to `data/processed/train.parquet` and `data/processed/test.parquet`.

In [1]:
import os
from pathlib import Path

# Mirror the same path convention used in 01_extract / 02_transform
path = os.getcwd()
path = os.path.abspath(os.path.join(path, "..", "data"))

DATA_DIR      = Path(path)
INTERIM_DIR   = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_FILE = INTERIM_DIR / "features.parquet"
TRAIN_FILE    = PROCESSED_DIR / "train.parquet"
TEST_FILE     = PROCESSED_DIR / "test.parquet"

TARGET_COLUMN = "label"
TEST_SIZE     = 0.20
RANDOM_STATE  = 42

print(f"Features source : {FEATURES_FILE}")
print(f"Train target    : {TRAIN_FILE}")
print(f"Test  target    : {TEST_FILE}")
print(f"test_size={TEST_SIZE}  random_state={RANDOM_STATE}  target='{TARGET_COLUMN}'")

Features source : /Users/marcel/Documents/github/property-near-the-beach-predictor/data/interim/features.parquet
Train target    : /Users/marcel/Documents/github/property-near-the-beach-predictor/data/processed/train.parquet
Test  target    : /Users/marcel/Documents/github/property-near-the-beach-predictor/data/processed/test.parquet
test_size=0.2  random_state=42  target='label'


## Load the feature matrix

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

assert FEATURES_FILE.exists(), (
    f"Feature matrix not found — run 02_transform first.\n{FEATURES_FILE}"
)

df = pd.read_parquet(FEATURES_FILE)
print(f"Loaded feature matrix: {df.shape[0]} rows × {df.shape[1]} columns")

# Guard against duplicate images leaking across the split boundary.
if "filepath" in df.columns:
    before = len(df)
    df = df.drop_duplicates(subset="filepath").reset_index(drop=True)
    if before != len(df):
        print(f"Dropped {before - len(df)} duplicate filepath(s) before splitting.")

df.head()

Loaded feature matrix: 27624 rows × 116 columns


,filepath,label,class_name,rgb_mean_r,rgb_mean_g,rgb_mean_b,rgb_std_r,rgb_std_g,rgb_std_b,hsv_mean_h,...,hist_b_30,hist_b_31,hue_red,hue_orange,hue_yellow,hue_green,hue_cyan,hue_blue,hue_indigo,hue_violet
0,data/raw/beach/i0001.jpg,1,beach,0.334138,0.477961,0.615541,0.353107,0.197764,0.142104,0.371241,...,0.000020,0.000000,0.207617,0.003917,0.002316,0.003260,0.762245,0.004146,0.003460,0.013039
1,data/raw/beach/i0002.jpg,1,beach,0.393676,0.613026,0.722836,0.333976,0.127429,0.103860,0.423970,...,0.014250,0.000399,0.193426,0.000000,0.000000,0.000000,0.806550,0.000000,0.000000,0.000024
2,data/raw/beach/i0003.jpg,1,beach,0.233265,0.528555,0.661930,0.296406,0.119145,0.100134,0.464619,...,0.000000,0.000000,0.080364,0.000000,0.000000,0.000799,0.918425,0.000242,0.000000,0.000170
3,data/raw/beach/i0004.jpg,1,beach,0.270062,0.602901,0.739404,0.312797,0.118020,0.136312,0.467375,...,0.037149,0.004265,0.067057,0.000000,0.000000,0.000252,0.930822,0.000454,0.000025,0.001389
4,data/raw/beach/i0005.jpg,1,beach,0.467254,0.607660,0.692338,0.330986,0.146475,0.130477,0.339171,...,0.000000,0.000000,0.145803,0.000143,0.000394,0.001468,0.835869,0.010703,0.000859,0.004761


## Split (stratified)

In [3]:
target = TARGET_COLUMN
stratify = df[target] if target in df.columns else None
if stratify is None:
    print(f"Warning: target '{target}' not found — splitting without stratification.")

train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratify,
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {train_df.shape[0]} rows  ({train_df.shape[0] / len(df):.1%})")
print(f"Test : {test_df.shape[0]} rows  ({test_df.shape[0] / len(df):.1%})")

Train: 22099 rows  (80.0%)
Test : 5525 rows  (20.0%)


## Sanity-check the split

In [4]:
# No overlap between train and test
if "filepath" in df.columns:
    overlap = set(train_df["filepath"]) & set(test_df["filepath"])
    assert not overlap, f"{len(overlap)} filepath(s) appear in both train and test!"
    assert len(train_df) + len(test_df) == len(df), "Row count mismatch after split"
    print("✓ No filepath overlap between train and test")
    print("✓ Row counts add up to the original")

if target in df.columns:
    dist = pd.DataFrame({
        "overall": df[target].value_counts(normalize=True),
        "train":   train_df[target].value_counts(normalize=True),
        "test":    test_df[target].value_counts(normalize=True),
    }).sort_index()
    print("\nTarget proportion by split (stratification preserves these):")
    print(dist.to_string(float_format=lambda x: f"{x:.3f}"))

✓ No filepath overlap between train and test
✓ Row counts add up to the original

Target proportion by split (stratification preserves these):
       overall  train  test
label                      
0        0.902  0.902 0.902
1        0.098  0.098 0.098


## Persist splits

In [5]:
train_df.to_parquet(TRAIN_FILE, index=False)
test_df.to_parquet(TEST_FILE, index=False)
print(f"Wrote {TRAIN_FILE}  ({train_df.shape[0]} rows × {train_df.shape[1]} cols)")
print(f"Wrote {TEST_FILE}  ({test_df.shape[0]} rows × {test_df.shape[1]} cols)")

Wrote /Users/marcel/Documents/github/property-near-the-beach-predictor/data/processed/train.parquet  (22099 rows × 116 cols)
Wrote /Users/marcel/Documents/github/property-near-the-beach-predictor/data/processed/test.parquet  (5525 rows × 116 cols)
